# CIFAR-10 Image Classifier

Convolutional neural network trained on CIFAR-10 (10 image classes, 32×32 RGB). Uses data augmentation, batch normalisation, and a learning-rate scheduler. Final model is saved to `models/cifar10_cnn.keras` for the website Demo.

## Part 0 — Setup

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from tensorflow.keras.datasets import cifar10

print('TensorFlow', tf.__version__)

### Class labels (CIFAR-10 has 10 categories)

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## Part 1 — Load & Prepare Data

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print('Train:', X_train.shape, '  Test:', X_test.shape)

### Normalise pixel values to [0, 1]

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

### Inspect a sample

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(class_names[int(y_train[i])])
    ax.axis('off')
plt.tight_layout()
plt.show()

## Part 2 — Build the Model

Three convolutional blocks → global average pooling → dense classifier. Augmentation layer is applied during training only.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    data_augmentation,

    # Block 1
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    # Block 2
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    # Block 3
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    # Head
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## Part 3 — Train

ReduceLROnPlateau drops the learning rate when validation loss stops improving. EarlyStopping cuts the run short if no progress is made. Tip: lower `EPOCHS` for a quick demo, raise it for accuracy.

In [ ]:
EPOCHS = 15  # bump to 30+ for higher accuracy

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=6, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    validation_data=(X_test, y_test),
    callbacks=[reduce_lr, early_stop],
    verbose=1,
)

### 3.1 Plot training history

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(history.history['accuracy'], label='train')
ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## Part 4 — Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss     : {test_loss:.4f}')
print(f'Test accuracy : {test_accuracy:.4f}')

### 4.1 Inspect predictions on test samples

In [ ]:
preds = model.predict(X_test[:10], verbose=0)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i])
    pred_class = class_names[int(np.argmax(preds[i]))]
    true_class = class_names[int(y_test[i])]
    ok = pred_class == true_class
    ax.set_title(f'{pred_class}\n(true: {true_class})',
                 color='green' if ok else 'red', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Part 5 — Persist the Model

Saved to `models/cifar10_cnn.keras` so the website backend can load it for the Demo.

In [ ]:
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / 'cifar10_cnn.keras'
model.save(MODEL_PATH)

print('Saved ->', MODEL_PATH, f'({MODEL_PATH.stat().st_size/1024:.1f} KB)')

## Part 6 — Prediction Helper

Reusable function that loads any image, resizes to 32×32, and returns a class label with confidence. The website backend imports this logic via `backend/predictors/cnn_cifar10.py`.

In [ ]:
from PIL import Image

def predict_image(img_or_path):
    """Accepts a file path, PIL.Image, or numpy array. Returns (label, confidence, all_probs)."""
    if isinstance(img_or_path, (str, Path)):
        img = Image.open(img_or_path).convert('RGB')
    elif isinstance(img_or_path, Image.Image):
        img = img_or_path.convert('RGB')
    else:
        img = Image.fromarray(np.asarray(img_or_path)).convert('RGB')

    img = img.resize((32, 32))
    arr = np.asarray(img, dtype='float32') / 255.0
    arr = np.expand_dims(arr, axis=0)
    probs = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), {c: float(p) for c, p in zip(class_names, probs)}

label, conf, _ = predict_image(X_test[0])
print(f'Test sample 0 -> {label} ({conf:.2%})')